# PZ data challenge: task set 2

**Task set 2: non-representative training.** The training set is emulated spectroscopic selections (DESI, DEEP2, VVDS, zCOSMOS), so it is much brighter than the test set, which keeps every object down to i < 25.4. There is no faint truth, so the held-out metrics are **optimistic** for the test set; the faintest held-out bin is the closest proxy.

This notebook only trains. For each of the four configurations of the task set (cardinal and flagship, 1yr and 10yr) it trains the CNN ensemble on the training file, holding out `VAL_FRAC` of it (split by `SEED`), and saves the trained ensemble as a single `.pkl` under its official name in `OUT_DIR` (the subtask 2 file, via `znn.save_ensemble_file`), shared by all four task-set notebooks. Everything that uses the models, the test-set p(z)s and the scores on the held-out objects, is in `pz_challenge_results.ipynb`, which predicts from these files.

The ensemble is trained with `znn.train_from_catalog`. Each member is trained on its own noise realization of the training photometry, redrawn from the catalogue's `_err` columns (the photometry as given if it has no errors), and on its own K-fold cut: member k validates on fold k and trains on the others. Every member starts from the pop-cosmos model shipped with znn. With `FROM_BASE = True` (the default), a base model is first trained on all the objects and each member is a copy of it fine-tuned on its realization and fold for `FINETUNE_EPOCHS` epochs at a constant `FINETUNE_LR`, which is cheaper per member.

Run this from the repo root with the `qp` conda env as the kernel. The filter curves are the ones shipped with RAIL, which `pz_data_challenge` installs.

## Configuration

In [ ]:
from rail.utils.path_utils import find_rail_file

import znn  # run from the repo root so this picks up the local source

TASKSET = 2
SIMS = ["cardinal", "flagship"]
SCENARIOS = ["1yr", "10yr"]

DATA_DIR = "../pz_data_challenge/scripts/public"
# Training starting point: "popcosmos" (the pre-training shipped with znn) or None (from scratch). Each choice
# writes to its own directory, so runs with and without pre-training never overwrite each other.
PRETRAINING = "popcosmos"
# How the ensemble is trained (znn.train_from_catalog): every member on its own noise realization of the
# training photometry (the photometry as given if it has no errors) and its own K-fold cut (member k validates
# on fold k and trains on the others)
# True: train one base model on all the objects, then fine-tune a copy of it per member (its own realization
# and fold) at FINETUNE_LR for FINETUNE_EPOCHS, cheaper per member; False: every member trained in full
FROM_BASE = True  # on task set 3 cardinal 10yr, better scatter and much better calibration (pz_challenge_split_test.ipynb)
# Method version, appended to OUT_DIR so runs of different versions never overwrite each other ("" for none)
RUN = "noisy_kfold" + ("_frombase" if FROM_BASE else "")
OUT_DIR = ("outputs/znn" if PRETRAINING is None else f"outputs/znn_pretrained_{PRETRAINING}") + (f"_{RUN}" if RUN else "")  # submission files for all task sets; the contents of the submission tarball

# Filter curves shipped with RAIL; the challenge's catalogs.yaml ("cardinal_roman_rubin") maps the
# catalogue columns to these DC2LSST_* and roman_* filters
FILTER_DIR = find_rail_file("examples_data/estimation_data/data/FILTER")

# band -> (catalogue magnitude column, filter curve file), ordered by wavelength
BANDS = {
    "u": ("mag_u_lsst", f"{FILTER_DIR}/DC2LSST_u.res"),
    "g": ("mag_g_lsst", f"{FILTER_DIR}/DC2LSST_g.res"),
    "r": ("mag_r_lsst", f"{FILTER_DIR}/DC2LSST_r.res"),
    "i": ("mag_i_lsst", f"{FILTER_DIR}/DC2LSST_i.res"),
    "z": ("mag_z_lsst", f"{FILTER_DIR}/DC2LSST_z.res"),
    "y": ("mag_y_lsst", f"{FILTER_DIR}/DC2LSST_y.res"),
    "Y": ("mag_Y_roman", f"{FILTER_DIR}/roman_Y106.res"),
    "J": ("mag_J_roman", f"{FILTER_DIR}/roman_J129.res"),
    "H": ("mag_H_roman", f"{FILTER_DIR}/roman_H158.res"),
}
REF_BAND = "i"  # magnitudes are normalized by this band
FEATURE_CONFIG = znn.DEFAULT_FEATURE_CONFIG

# Photometric errors sampled over in training: magnitude column -> its error column
ERRORS = {col: f"{col}_err" for col, _ in BANDS.values()}
NOISE_SEED = 42  # seed of the training noise realizations

VAL_FRAC = 0.2  # fraction of the training set held out for evaluation (in pz_challenge_results.ipynb, which must match)
N_MEMBERS = 10 if FROM_BASE else 5  # ensemble members, and K-fold folds
EPOCHS = 100  # max epochs per member, or of the base model with FROM_BASE (early stopping usually ends sooner)
FINETUNE_EPOCHS = 10  # FROM_BASE: epochs of each member's fine-tuning (no early stopping)
FINETUNE_LR = 3e-4  # FROM_BASE: constant fine-tuning learning rate (1e-3 gives better calibration, slightly worse scatter)
SEED = 42

In [ ]:
import os

import numpy as np
from sklearn.model_selection import train_test_split

os.makedirs(OUT_DIR, exist_ok=True)
print("znn from", znn.__file__)

## Reference redshifts

In [ ]:
def reference_redshifts(train):
    """Reference redshift per training object, and the held-out subsets to score."""
    z_ref = train["redshift"].to_numpy()  # every training object has a spec-z
    assert np.isfinite(z_ref).all()
    subsets = {
        "held-out (spec-selected)": np.ones(len(train), dtype=bool),
        "held-out i > 23 (faintest available)": train["mag_i_lsst"].to_numpy() > 23,
    }
    return z_ref, subsets

## Run all configurations

For each configuration: train on the fit split and save the model file. The held-out split is left out of training so `pz_challenge_results.ipynb` can score on it.

In [ ]:
for SIM in SIMS:
    for SCENARIO in SCENARIOS:
        config = f"{SIM}_{SCENARIO}"
        print(f"\n========== task set {TASKSET}: {config} ==========")

        train_file = os.path.join(DATA_DIR, f"pz_challenge_taskset_{TASKSET}_{SIM}_training_{SCENARIO}.hdf5")
        train = znn.read_catalog(train_file)
        z_ref, _ = reference_redshifts(train)

        rng = np.random.default_rng(NOISE_SEED)
        # the held-out split is not used here, only kept out of training (pz_challenge_results.ipynb scores on it)
        idx_fit, idx_val = train_test_split(np.arange(len(train)), test_size=VAL_FRAC, random_state=SEED)
        trained_models, histories, base_history = znn.train_from_catalog(
            znn.build_model, train.iloc[idx_fit], z_ref[idx_fit], BANDS, REF_BAND, FEATURE_CONFIG, errors=ERRORS, rng=rng,
            N_MEMBERS=N_MEMBERS, from_base=FROM_BASE, EPOCHS=EPOCHS, FINETUNE_EPOCHS=FINETUNE_EPOCHS,
            FINETUNE_LR=FINETUNE_LR, random_state=SEED, pretraining=PRETRAINING,
        )
        if base_history is not None:
            znn.plot_ensemble_losses([base_history], ylim=(0, 0.1))
        znn.plot_ensemble_losses(histories, ylim=(0, 0.1))

        # subtask 2 submission file: the trained ensemble
        model_file = os.path.join(OUT_DIR, f"pz_challenge_taskset_{TASKSET}_{SIM}_pz_model_{SCENARIO}.pkl")
        znn.save_ensemble_file(model_file, trained_models, BANDS, REF_BAND, FEATURE_CONFIG)
        print("saved", model_file)